# AETHER agent_c supporting execution

Development-only; requires reviewed committed config.

In [ ]:
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,tarfile
WORK=Path('/kaggle/working'); RUNTIME=Path('/tmp/aether-agent_c'); REPO=RUNTIME/'aether'
VENV=RUNTIME/'venv'; PY=VENV/'bin'/'python'; OUTPUT=WORK/'aether-agent_c-results'
COMMIT='7ffad6349aae998ed94cfaa526fbfe103549926f'; URL='https://github.com/ama1a005/aether.git'
RUNTIME.mkdir(parents=True,exist_ok=True); os.environ['PYTHONNOUSERSITE']='1'
if shutil.which('uv') is None: subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
UV=shutil.which('uv'); env=os.environ.copy(); env['UV_PYTHON_INSTALL_DIR']=str(RUNTIME/'uv-python')
subprocess.run([UV,'venv','--python','3.11.9',str(VENV)],check=True,env=env)
subprocess.run(['git','clone','--branch','phase1/core-nov19','--single-branch',URL,str(REPO)],check=True)
subprocess.run(['git','fetch','origin',COMMIT],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==COMMIT
subprocess.run([UV,'pip','install','--python',str(PY),'-r',str(REPO/'requirements-lock.txt')],check=True,env=env)
subprocess.run([UV,'pip','install','--python',str(PY),'--no-deps','-e',str(REPO)],check=True,env=env)


In [ ]:
subprocess.run([str(PY),'-m','experiments.phase1.qwen_agent_c','--config',str(REPO/'configs/experiments/agent_c_qwen_development.yaml'),'--output',str(OUTPUT)],cwd=REPO,check=True)

In [ ]:
subprocess.run([str(PY),'-m','scripts.verify_external_bundle',str(OUTPUT)],cwd=REPO,check=True)
archive=WORK/'aether-agent_c-results.tar.gz'
with tarfile.open(archive,'w:gz') as bundle: bundle.add(OUTPUT,arcname=OUTPUT.name)
print(archive,hashlib.sha256(archive.read_bytes()).hexdigest())